In [1]:
import json
import os
import networkx as nx
from plan import PartialPlan
from old.run_mzn import run_mzn
from old.mzn_arr_to_schedule import *
from objects import Service, Movement

In [2]:
from parse_planner_output import extract_metricff_plan
from schedule_to_plan import extract_info, make_new_schedule, make_new_new_schedule

In [3]:
walking_distances = {
    ("entry", "52"):3,
    ("entry", "53"):4,
    ("entry", "54"):5,
    ("entry", "55"):6,
    ("entry", "56"):7,
    ("entry", "57"):8,
    ("entry", "58"):9,
    ("entry", "59"):10,
    ("entry", "60"):8,
    ("entry", "61_service"):9,
    ("entry", "62_service"):10,
    ("entry", "63"):12,
    ("52", "53"):1,
    ("52", "54"):2,
    ("52", "55"):3,
    ("52", "56"):4,
    ("52", "57"):5,
    ("52", "58"):6,
    ("52", "59"):7,
    ("52", "60"):5,
    ("52", "61_service"):6,
    ("52", "62_service"):7,
    ("52", "63"):10,
    ("53", "54"):1,
    ("53", "55"):2,
    ("53", "56"):3,
    ("53", "57"):4,
    ("53", "58"):5,
    ("53", "59"):6,
    ("53", "60"):4,
    ("53", "61_service"):5,
    ("53", "62_service"):6,
    ("53", "63"):9,
    ("54", "55"):1,
    ("54", "56"):2,
    ("54", "57"):3,
    ("54", "58"):4,
    ("54", "59"):5,
    ("54", "60"):3,
    ("54", "61_service"):4,
    ("54", "62_service"):5,
    ("54", "63"):8,
    ("55", "56"):1,
    ("55", "57"):2,
    ("55", "58"):3,
    ("55", "59"):4,
    ("55", "60"):4,
    ("55", "61_service"):3,
    ("55", "62_service"):4,
    ("55", "63"):7,
    ("56", "57"):1,
    ("56", "58"):2,
    ("56", "59"):3,
    ("56", "60"):5,
    ("56", "61_service"):4,
    ("56", "62_service"):3,
    ("56", "63"):6,
    ("57", "58"):1,
    ("57", "59"):2,
    ("57", "60"):6,
    ("57", "61_service"):5,
    ("57", "62_service"):4,
    ("57", "63"):7,
    ("58", "59"):1,
    ("58", "60"):7,
    ("58", "61_service"):6,
    ("58", "62_service"):5,
    ("58", "63"):8,
    ("59", "60"):8,
    ("59", "61_service"):7,
    ("59", "62_service"):6,
    ("59", "63"):9,
    ("60", "61_service"):1,
    ("60", "62_service"):2,
    ("60", "63"):4,
    ("61_service", "62_service"):1,
    ("61_service", "63"):3,
    ("62_service", "63"):4,
}

In [4]:
rows = list()
for cfg in range(17, 50):
    for num_t in range(3, 16):

        plan_file = f"../results/metricff/base3/pln_{cfg}_{num_t}t.txt"

        if not os.path.exists(plan_file):
            continue
        
        plans, costs, search_time = extract_metricff_plan(plan_file)
        
        solved = len(plans) > 0
        if solved:
            plan_lines = [f'({l.split(': ')[-1][:-1].lower()})' for l in plans[0]]
            pp = PartialPlan(plan_lines)
            pp.build_constraints()
            pp.build_walking_times_matrix(walking_distances)
            pp.write_dzn(1)

            [start_times, durations, action_driver, action_train] = run_mzn(300, 'chuffed')
            train_schedule, driver_schedule = init_train_driver_schedules(start_times,durations, 
                                                                            action_train, action_driver)

            new_schedule = make_new_schedule(pp, start_times)
            new_new_schedule = make_new_new_schedule(new_schedule)

            with open(f'../results/metricff/base3/plans/plan_{cfg}_{num_t}t', 'w') as f:
                f.writelines([str(l)+'\n' for l in new_new_schedule])

            pp_dur = max([start_times[i]+durations[i] for i in range(len(start_times))])

            plot_schedule2(train_schedule, pp.actions, f'../results/metricff/base3/plots/plot_{cfg}_{num_t}t')

            rows.append({'config':cfg,'num_trains':num_t,'cost':costs[0],'makespan_pp':int(pp_dur),'search_time':search_time})

df = pd.DataFrame(rows)
if not os.path.exists('results_base3_metricff.csv'):
    df.to_csv('results_base3_metricff.csv', index=False)
else:
    df_old = pd.read_csv('results_base3_metricff.csv')
    df_new = pd.concat([df,df_old], ignore_index=True)
    df_new = df_new.drop_duplicates(subset=['config','num_trains'])
    df_new.to_csv('results_base3_metricff.csv', index=False)

In [5]:
# cfg_list = list(range(7,50))
# cfg_list.reverse()
# for cfg in cfg_list:
#     for num_t in range(3,16):
#         plan_file = f"../results/enhsp/base3/pln_{cfg}_{num_t}t.txt"
#         if not os.path.exists(plan_file):
#             # rows.append({'config':cfg,'num_trains':num_t,'solved':False,'num_expansions':None,'makespan':None})
#             print('not exists')
#             continue
#         os.rename(plan_file, f"../results/enhsp/base3/pln_{cfg+1}_{num_t}t.txt")